# 퍼뮤테이션 검정 

귀무가설 하에서의 검정 통계량 분포를 직접 생성하기 위해,  
데이터를 무작위로 여러 번 섞어(permutation) 그때의 통계량들을 비교하는 비모수적 검정 방법.  
- 귀무가설 : 두 그룹은 같은 분포에서 나왔다 (차이 없다)
- 통계량 : 평균 차이, t값, 상관계수 등
- 방법 : 관측값의 레이블을 무작위로 섞어서 통계량 재계산 → 이걸 수천 번 반복

### 장점
- 정규성, 등분산성 같은 가정 불필요
- 유연하게 다양한 통계량에 적용 가능
- 표본 수가 작아도 강력함
- 정확한 분포 대신 경험적 분포 사용
### 단점 
- 계산량 많음 (반복횟수 많을수록 정확)
- 해석 시 무작위성 반영 필요


In [1]:
# 예제문제

#두 학습 그룹 A, B의 수학 점수를 비교하려고 한다.
#정규성이나 등분산성 가정을 하지 않고, 관측된 평균 차이가 유의미한지를 퍼뮤테이션 검정으로 판단하라.

In [4]:
import numpy as np

np.random.seed(42)

# 두 그룹의 실제 점수
group_A = np.random.normal(loc=75, scale=5, size=10)
group_B = np.random.normal(loc=80, scale=5, size=10)

print(" A그룹 : ",group_A)
print(" B그룹 : ",group_B)

 A그룹 :  [77.48357077 74.30867849 78.23844269 82.61514928 73.82923313 73.82931522
 82.89606408 78.83717365 72.65262807 77.71280022]
 B그룹 :  [77.68291154 77.67135123 81.20981136 70.43359878 71.37541084 77.18856235
 74.9358444  81.57123666 75.45987962 72.93848149]


In [5]:
# 실제 관측된 평균 차이
obs_diff = np.mean(group_A) - np.mean(group_B)

# 퍼뮤테이션 검정
combined = np.concatenate([group_A, group_B])
n_permutations = 10000
perm_diffs = []

for _ in range(n_permutations):
    np.random.shuffle(combined)
    new_A = combined[:10]
    new_B = combined[10:]
    diff = np.mean(new_A) - np.mean(new_B)
    perm_diffs.append(diff)

# p-value 계산 (양측)
perm_diffs = np.array(perm_diffs)
p_value = np.mean(np.abs(perm_diffs) >= np.abs(obs_diff))

# 출력
print("퍼뮤테이션 검정 결과")
print(f"관측된 평균 차이: {obs_diff:.4f}")
print(f"p-value: {p_value:.4f}")  # 귀무가설을 채택하여 관측된 두 집단간의 평균차이가 없다.

퍼뮤테이션 검정 결과
관측된 평균 차이: 1.1936
p-value: 0.4739


# 실제로 쓰이는 분야 
1. 피어슨 상관계수의 유의성 검정 (Permutation Test)

- 두 연속형 변수 X와 Y의 상관관계가 우연히 나타난 것이 아닌지 검정
- 귀무가설 H₀: X와 Y는 독립이다 (r = 0)

In [6]:
import numpy as np
from scipy.stats import pearsonr

np.random.seed(1)

# 예제 데이터
x = np.random.normal(0, 1, 30)
y = 2 * x + np.random.normal(0, 1, 30)  # 강한 선형 관계

# 실제 상관계수
r_obs, _ = pearsonr(x, y)

# 퍼뮤테이션 검정
n_perms = 10000
r_perms = []

for _ in range(n_perms):
    y_perm = np.random.permutation(y)
    r_perm, _ = pearsonr(x, y_perm)
    r_perms.append(r_perm)

# p-value 계산 (양측 검정)
r_perms = np.array(r_perms)
p_value = np.mean(np.abs(r_perms) >= np.abs(r_obs))

print(f"p-value: {p_value:.4f}")  # 귀무가설을 기각하여 상관관계는 유의미하다. / 대립가설 : 선형관계가 우연히 나타날 가능성이 있다.

p-value: 0.0000


2. 회귀 계수의 유의성 평가 (Permutation Test)

- 독립변수 X가 종속변수 Y에 유의미한 영향을 미치는지
- 귀무가설 H₀: X의 회귀계수 β = 0

In [8]:
import pandas as pd
import statsmodels.api as sm

np.random.seed(2)

# 데이터 생성
x = np.random.normal(0, 1, 30)
y = 3 * x + np.random.normal(0, 1, 30) # 실제 계수는 0이 아님

# 실제 회귀계수
X = sm.add_constant(x)
model = sm.OLS(y, X).fit()
beta_obs = model.params[1]

# 퍼뮤테이션 검정
beta_perms = []
n_perms = 10000

for _ in range(n_perms):
    y_perm = np.random.permutation(y)
    model_perm = sm.OLS(y_perm, X).fit()
    beta_perms.append(model_perm.params[1])

# p-value 계산 (양측 검정)
beta_perms = np.array(beta_perms)
p_value_beta = np.mean(np.abs(beta_perms) >= np.abs(beta_obs))

print("\n회귀계수 퍼뮤테이션 검정")
print(f"관측된 회귀계수 β: {beta_obs:.4f}")
print(f"p-value: {p_value_beta:.4f}")  # 귀무가설 기각하여 회귀계수는 유의미하다. / 대립가설 : 회귀계수= 0이 우연히 나타날 가능성이 있다.


회귀계수 퍼뮤테이션 검정
관측된 회귀계수 β: 3.1887
p-value: 0.0000
